# Zalo AI Challenge 2026 — LifeMate (ReAct + Schema-Grounded QLoRA)

Thin-runner notebook for Google Colab (T4 16GB):
- **Step 1:** Sync repo `duylhm/ZaloAIC-Lifemate`, mount Drive, unzip data
- **Step 2 (Optional):** Benchmark Vietnamese ASR models on 50 `train` WAVs
- **Step 3:** Zero-shot + QLoRA Validation on Unseen-App CV (`142` unseen + `100` in-domain)
- **Step 4:** Train Full 1,545 samples -> Infer `public_test` (634) -> Validate & Download `submission.jsonl`

In [ ]:
import os, pathlib, subprocess

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = pathlib.Path("/content/ZaloAIC-Lifemate")
if not REPO_DIR.exists():
    !git clone https://github.com/duylhm/ZaloAIC-Lifemate.git /content/ZaloAIC-Lifemate
else:
    !git -C /content/ZaloAIC-Lifemate pull --ff-only

%cd /content/ZaloAIC-Lifemate
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

DATA_DIR = pathlib.Path("/content/data")
ZIP_PATH = pathlib.Path("/content/drive/MyDrive/train+public_test.zip")
if not (DATA_DIR / "train").exists() and not (DATA_DIR / "train+public_test" / "train").exists():
    if ZIP_PATH.is_file():
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        !unzip -q -o "{ZIP_PATH}" -d /content/data
    else:
        print("WARNING: Update ZIP_PATH if train+public_test.zip is elsewhere on MyDrive.")

!python scripts/prepare_splits.py

In [ ]:
# Choose base SLM (<5B params): "Qwen/Qwen3-4B" (with /no_think) or "Qwen/Qwen2.5-3B-Instruct"
MODEL_ID = "Qwen/Qwen3-4B"
MODEL_TAG = MODEL_ID.split("/")[-1]
CKPT_ROOT = f"/content/drive/MyDrive/lifemate_checkpoints/{MODEL_TAG}"
print(f"Active MODEL_ID={MODEL_ID} | Checkpoints={CKPT_ROOT}")

In [ ]:
# [Optional] Benchmark Vietnamese ASR CER/WER on 50 train WAVs (uncomment whisper-large-v3-turbo if needed)
!python scripts/benchmark_asr.py --models vinai/PhoWhisper-large openai/whisper-large-v3-turbo --limit 50 --num-beams 3

In [ ]:
# [Optional Baseline] Zero-shot Unseen-App CV score on 242 val samples
!python run_inference.py \
  --split train \
  --ids-file splits/val_all.json \
  --model "{MODEL_ID}" \
  --force \
  --output outputs/val_zeroshot.jsonl \
  --eval

In [ ]:
# Phase B: Fine-tune QLoRA on train_ids (1,303) & validate on val_unseen (142) + val_indomain (100)
VAL_ADAPTER = f"{CKPT_ROOT}_val"
!python train_sft.py \
  --model "{MODEL_ID}" \
  --split-mode val \
  --epochs 2 \
  --aug-copies 1 \
  --output-dir "{VAL_ADAPTER}"

!python run_inference.py \
  --split train \
  --ids-file splits/val_all.json \
  --model "{MODEL_ID}" \
  --adapter "{VAL_ADAPTER}" \
  --force \
  --output outputs/val_sft.jsonl \
  --eval

!python scripts/inspect_errors.py \
  --split train \
  --ids-file splits/val_all.json \
  --submission outputs/val_sft.jsonl \
  --max-per-bucket 5

In [ ]:
# Phase C: Train full QLoRA on all 1,545 samples -> infer public_test (634) -> validate & backup
FULL_ADAPTER = f"{CKPT_ROOT}_full"
!python train_sft.py \
  --model "{MODEL_ID}" \
  --split-mode full \
  --epochs 2 \
  --aug-copies 1 \
  --output-dir "{FULL_ADAPTER}"

!python run_inference.py \
  --split public_test \
  --model "{MODEL_ID}" \
  --adapter "{FULL_ADAPTER}" \
  --force \
  --output /content/submission.jsonl \
  --mirror-path /content/drive/MyDrive/submission.jsonl

from google.colab import files
files.download("/content/submission.jsonl")